# 07 — Dashboard Insights (Wait-Time Model → Admin Dashboards)

Trains the **GradientBoostingRegressor** wait-time model (same features as
notebook 05: day-of-week, hour, branch, service, queue length at join) and
emits the structured insights the admin dashboards chart directly:

| insight_type | What the dashboard shows |
|---|---|
| `wait_time_predictions` | Predicted wait per hour (8:00–17:00) under *Analytics* |
| `abandonment_thresholds` | Queue length at which people stop joining, per service |
| `model_performance` | Holdout MAE / R² so executives can judge trust |

Set `WRITE_DB=1` to upsert straight into `predictive_results`; otherwise the
insights are written to `outputs/admin/dashboard_insights.json` for
`scripts/import_predictions.py` to push through the API pipeline.

In [ ]:
import os, sys
from pathlib import Path

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(BASE))

from scripts import generate_insights as gi

conn = gi.connect()
df = gi.load_records(conn)
print(f'{len(df)} visit records loaded')

trained = gi.train_model(df)
assert trained, 'Not enough completed visits to train the model.'
print(f"Model quality — MAE ±{trained['mae']:.1f}m, R² {trained['r2']:.2f} ({trained['test_rows']} holdout rows)")

In [ ]:
insights, generated_at, stale_after = gi.build_insights(df, trained)
path = gi.write_outputs(insights)
print(f'{len(insights)} insights written to {path}')

if os.getenv('WRITE_DB') == '1':
    gi.write_db(conn, insights, generated_at, stale_after, len(df))
    print('Upserted into predictive_results.')
conn.close()